In [ ]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
torch.set_printoptions(sci_mode=False,precision=4)
torch.manual_seed(123)
torch.random.manual_seed(123)
generator = torch.Generator().manual_seed(123)


In [ ]:
with open('../data/numeric_data.txt', 'r', encoding = 'utf-8') as file:
    names = file.read()

In [ ]:
context_size = 36
embd_dim = 32
logging = False

device = torch.device("cpu" if torch.backends.mps.is_available() else "cpu")
batch_size = 64 if device.type == "mps" else 64
print(device, "| batch_size (before clamp):", batch_size)

In [ ]:
data = ''.join(names)[0:100000]
data = '_' * context_size + data
unique_text = sorted(list(set(data)))
vocab_size = len(unique_text)

# print(''.join(unique_text))
# print(len(''.join(unique_text)))

stoi = {text: index for index, text in enumerate(unique_text)}
itos = {index: text for index, text in enumerate(unique_text)}

encoder = lambda char_array: [stoi[char] for char in ''.join(char_array)]
decoder = lambda num_array: [itos[num] for num in num_array]

In [ ]:
print(stoi)

In [ ]:
x_numpy = []
y_numpy = []
temp_data = data

for t in range(len(temp_data) - context_size - 1):
    x_numpy.append(encoder(temp_data[t : t + context_size]))
    y_numpy.append(stoi[temp_data[t + context_size]])

x = torch.tensor(x_numpy, dtype=torch.long).to(device)
y = torch.tensor(y_numpy, dtype=torch.long).to(device)

N = x.shape[0]
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

train_end = int(train_ratio * N)
val_end = int((train_ratio + val_ratio) * N)

x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:val_end], y[train_end:val_end]
x_test, y_test = x[val_end:], y[val_end:]

x, y = x.to(device), y.to(device)

batch_size = min(batch_size, x.shape[0])

In [ ]:
# from torch import nn
# import torch.optim as optim
# _batch_size = 2
# _vocab_size = 100
# _embed_dim = 2
# embedding = nn.Embedding(_vocab_size, _embed_dim)
# x = torch.randint(0, 100, size=(_batch_size, 2), dtype=torch.long, generator=generator).to(device)
#
# emb = embedding(x)
# emb = emb.view(-1, _embed_dim)
#
# T = emb.shape[0]
# mat = torch.ones(T, T)
# upper = torch.tril(mat)
# upper = upper/upper.sum(1, keepdim=True)
# calc = upper @ emb
#
# # print(embedding)
# # print(x)
# # print(embedding(x))

In [ ]:
from torch import nn
import torch.optim as optim

class Model(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(context_size, context_size))
        )
        self.linear_1 = nn.Linear(embd_dim * context_size, vocab_size, bias=False)

    def forward(self, x_):
        B, T = x_.shape
        # calc = self.embedding(x_)
        emb = self.embedding(x_)
        tril_mask = self.tril[:T, :T]
        weights = tril_mask / tril_mask.sum(1, keepdim=True)
        calc = weights @ emb
        calc = calc.view(calc.size(0), -1)
        logits = self.linear_1(calc)
        return logits

In [ ]:
loss = 0
criterion = nn.CrossEntropyLoss()
model = Model(embd_dim).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

lr_iterations = []
lr_losses = []
iteration_count = 0
model.train()
for epoch in range(2000000):

    idx = torch.randint(0, len(x_train), (batch_size,), device=device, generator=generator)
    rand_x = x_train[idx]
    rand_y = y_train[idx]

    optimizer.zero_grad(set_to_none=True)
    outputs = model(rand_x)
    loss = criterion(outputs, rand_y)

    if epoch % 10000 == 0:
        print(loss.item())

    loss.backward()
    optimizer.step()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

hidden_layers = [4, 8, 16, 32]

criterion = nn.CrossEntropyLoss()
results = []

for hidden_layer in hidden_layers:

    print(f"\nTraining with hidden_layer = {hidden_layer}")

    model = Model(embd_dim, hidden_layer).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()

    for epoch in range(1000000):   # reduce for faster experimentation
        idx = torch.randint(0, len(x_train), (batch_size,), device=device)
        rand_x = x_train[idx]
        rand_y = y_train[idx]

        optimizer.zero_grad(set_to_none=True)
        outputs = model(rand_x)
        loss = criterion(outputs, rand_y)
        loss.backward()
        optimizer.step()

    model.eval()

    with torch.no_grad():
        train_loss = criterion(model(x_train), y_train).item()
        val_loss   = criterion(model(x_val), y_val).item()
        test_loss  = criterion(model(x_test), y_test).item()

    print(f"Hidden: {hidden_layer} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Test: {test_loss:.4f}")

    results.append({
        "hidden": hidden_layer,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "test_loss": test_loss
    })

In [ ]:
first = model.embedding.weight[:, 0].detach().float().cpu().numpy()
second = model.embedding.weight[:, 1].detach().float().cpu().numpy()
for i in range(len(first)):
    word = itos[i]
    # if word in ['h','e','a','r', ' ', 'm', 'e','p','3']:
    plt.scatter(first[i], second[i], s=500)
    plt.text(first[i], second[i], word, fontsize=10, ha='center', va='center')

In [ ]:
plt.plot(lr_iterations, lr_losses)
plt.show()

In [ ]:
# print(vocab_size)
torch.set_printoptions(sci_mode=False,precision=4)
print(model.embedding.weight.float())
print(model.linear_2.weight.cpu().detach())
for name, param in model.named_parameters():
    print(f"{name}: {param.numel()}")

In [ ]:
#Test
model.eval()
logging = True
final_text = '___________________________Would you proceed '.lower()
for _ in range(200):
    x = torch.tensor(encoder(final_text[-context_size:]))
    x = torch.unsqueeze(x,0)
    output = model(x)
    final = torch.max(output, 1)
    # print(final)
    y = decoder([final[1].item()])
    final_text+=y[0]
print(final_text)

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Get embeddings (assuming vocab_size >= 20)
embeddings = model.embedding.weight.detach().numpy()

# Reduce 10D → 2D
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

# Plot all 20 points in ONE scatter
plt.figure(figsize=(8, 8))

# Label each point (0–19)
for i in range(embeddings.shape[0]):
    word = itos[i]
    if word in ['h','e','a','r', ' ', 'm', 'e','s', '3']:
        plt.scatter(reduced[i, 0], reduced[i, 1], s=100)
        plt.text(reduced[i, 0], reduced[i, 1], word, ha='center', va='center')

plt.title("20 Embeddings Visualization (PCA)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()

In [ ]:
a = torch.tensor([ 1.2496,  4.0898, -0.5284,  1.0659, -2.6788,  0.3610, -1.3946,  2.2194,
         -1.9383, -0.1370,  0.0481,  2.6672, -0.5284,  1.0659, -1.9383, -0.1370])

b = torch.tensor([  3.8696,   0.7342,   3.6664,  -0.3347,  -0.6673,  -2.9894,  -1.8589,
           1.7606,   2.5973,  -6.5067,   0.9466,   0.7771,   1.5486,  -1.4092,
           0.7132,  -3.9513])

a = torch.unsqueeze(a, 0)
b = torch.unsqueeze(b, 1)

print(a.shape)
print(b.shape)
print(a @ b)